In [14]:
# ==========================================================
# Task 1: Web Scraping (TechReads Retail Analytics)
# Source: Goodreads "data-engineering" shelf
# Steps:
# 1) Collect book URLs from shelf page(s)
# 2) Visit each book page to extract Title, Author, Year, Rating, Price
# 3) Save to techreads_books.csv
# ==========================================================

import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

print("Libraries imported successfully.")

Libraries imported successfully.


In [15]:
# Use a browser-like User-Agent to reduce blocking.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-GB,en;q=0.9",
}

def fetch_html(url: str, timeout: int = 20) -> str:
    """Fetch HTML safely with headers."""
    resp = requests.get(url, headers=HEADERS, timeout=timeout)
    resp.encoding = "utf-8"
    return resp.text, resp.status_code

def looks_blocked(html: str) -> bool:
    """Basic detection for CAPTCHA/robot blocks."""
    t = html.lower()
    return ("captcha" in t) or ("robot" in t and "check" in t) or ("automated access" in t)

def extract_year(text: str) -> str:
    """Extract first 4-digit year from text."""
    m = re.search(r"\b(19|20)\d{2}\b", text)
    return m.group(0) if m else "N/A"

def extract_price(text: str) -> str:
    """
    Extract a price if present on the page (e.g., 'Kindle $42.99' or '£31.50').
    Returns 'N/A' if none found.
    """
    # Capture £xx.xx or $xx.xx
    m = re.search(r"([£$])\s?(\d+(?:\.\d{2})?)", text)
    if m:
        return f"{m.group(1)}{m.group(2)}"
    return "N/A"

print("Helpers ready.")

Helpers ready.


In [16]:
# Goodreads shelf page (you can change this URL if needed)
SHELF_URL = "https://www.goodreads.com/shelf/show/data-engineering"

TARGET_BOOKS = 15          # minimum required
MAX_PAGES = 5              # keep low to avoid blocks
POLITE_DELAY = 1.0         # seconds between requests

book_urls = []
seen = set()

for page in range(1, MAX_PAGES + 1):
    url = SHELF_URL if page == 1 else f"{SHELF_URL}?page={page}"
    html, status = fetch_html(url)

    if status != 200:
        print(f"Shelf page {page} failed: HTTP {status}")
        break
    if looks_blocked(html):
        print("Blocked/CAPTCHA detected on shelf page. Try again later or reduce requests.")
        break

    soup = BeautifulSoup(html, "html.parser")

    # Find links to book pages
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/book/show/" in href:
            full = urljoin("https://www.goodreads.com", href.split("?")[0])
            if full not in seen:
                seen.add(full)
                book_urls.append(full)

        if len(book_urls) >= TARGET_BOOKS:
            break

    print(f"Shelf page {page}: collected {len(book_urls)} book URLs so far")

    if len(book_urls) >= TARGET_BOOKS:
        break

    time.sleep(POLITE_DELAY)

print("\nTotal book URLs collected:", len(book_urls))
book_urls[:5]

Shelf page 1: collected 15 book URLs so far

Total book URLs collected: 15


['https://www.goodreads.com/book/show/61218623-fundamentals-of-data-engineering',
 'https://www.goodreads.com/book/show/23463279-designing-data-intensive-applications',
 'https://www.goodreads.com/book/show/748203.The_Data_Warehouse_Toolkit',
 'https://www.goodreads.com/book/show/55841851-data-pipelines-pocket-reference',
 'https://www.goodreads.com/book/show/44647144-database-internals']

In [17]:
# Visit each book page to extract: Title, Author, Year, Rating, RatingsCount
rows = []

for i, url in enumerate(book_urls, start=1):
    html, status = fetch_html(url)
    if status != 200:
        print(f"[{i}] Failed: HTTP {status} -> {url}")
        continue
    if looks_blocked(html):
        print(f"[{i}] Blocked/CAPTCHA detected -> stopping early")
        break

    soup = BeautifulSoup(html, "html.parser")
    page_text = soup.get_text(" ", strip=True)

    # Title
    title = "N/A"
    h1 = soup.find("h1")
    if h1:
        title = h1.get_text(" ", strip=True)

    # Author (new Goodreads UI)
    author = "N/A"
    author_span = soup.select_one('span.ContributorLink__name[data-testid="name"]')
    if author_span and author_span.get_text(strip=True):
        author = author_span.get_text(" ", strip=True)
    else:
        author_link = soup.find("a", href=re.compile(r"^/author/show/"))
        if author_link:
            author = author_link.get_text(" ", strip=True)

    # Year
    year = extract_year(page_text)

    # Rating (average rating)
    rating = "N/A"
    rating_candidates = re.findall(r"\b([0-5]\.\d{1,2})\b", page_text)
    if rating_candidates:
        rating = float(rating_candidates[0])

    # Ratings count (popularity indicator – useful for "trending")
    # Example text on page: "10,687 ratings 947 reviews"
    ratings_count = "N/A"
    m_rc = re.search(r"([\d,]+)\s+ratings", page_text.lower())
    if m_rc:
        ratings_count = int(m_rc.group(1).replace(",", ""))

    # Optional: Reviews count
    reviews_count = "N/A"
    m_rev = re.search(r"([\d,]+)\s+reviews", page_text.lower())
    if m_rev:
        reviews_count = int(m_rev.group(1).replace(",", ""))

    rows.append({
        "Title": title,
        "Author": author,
        "Year": year,
        "Rating": rating,
        "RatingsCount": ratings_count,   # ✅ replaces Price
        "ReviewsCount": reviews_count,   # optional extra field
        "Source": "goodreads",
        "URL": url
    })

    print(f"[{i}] OK: {title[:55]} | {author} | rating={rating} | ratings={ratings_count}")
    time.sleep(POLITE_DELAY)

df = pd.DataFrame(rows)
print("\nRows extracted:", len(df))
df.head(15)

[1] OK: Fundamentals of Data Engineering: Plan and Build Robust | Joe Reis | rating=4.18 | ratings=927
[2] OK: Designing Data-Intensive Applications | Martin Kleppmann | rating=4.7 | ratings=10687
[3] OK: The Data Warehouse Toolkit: The Complete Guide to Dimen | Ralph Kimball | rating=4.18 | ratings=1035
[4] OK: Data Pipelines Pocket Reference: Moving and Processing  | James Densmore | rating=3.77 | ratings=242
[5] OK: Database Internals: A deep-dive into how distributed da | Alex Petrov | rating=4.26 | ratings=558
[6] OK: Data Mesh: Delivering Data-Driven Value at Scale | Zhamak Dehghani | rating=3.79 | ratings=380
[7] OK: Data Pipelines with Apache Airflow | Bas P. Harenslak | rating=4.31 | ratings=118
[8] OK: Spark: The Definitive Guide: Big Data Processing Made S | Bill Chambers | rating=4.14 | ratings=283
[9] OK: Kafka: The Definitive Guide: Real-Time Data and Stream  | Neha Narkhede | rating=4.15 | ratings=727
[10] OK: Making Sense of Stream Processing | Martin Kleppmann | rating

,Title,Author,Year,Rating,RatingsCount,ReviewsCount,Source,URL
0,Fundamentals of Data Engineering: Plan and Bui...,Joe Reis,2022,4.18,927,97,goodreads,https://www.goodreads.com/book/show/61218623-f...
1,Designing Data-Intensive Applications,Martin Kleppmann,2015,4.70,10687,947,goodreads,https://www.goodreads.com/book/show/23463279-d...
2,The Data Warehouse Toolkit: The Complete Guide...,Ralph Kimball,1996,4.18,1035,65,goodreads,https://www.goodreads.com/book/show/748203.The...
3,Data Pipelines Pocket Reference: Moving and Pr...,James Densmore,2021,3.77,242,31,goodreads,https://www.goodreads.com/book/show/55841851-d...
4,Database Internals: A deep-dive into how distr...,Alex Petrov,2019,4.26,558,69,goodreads,https://www.goodreads.com/book/show/44647144-d...
5,Data Mesh: Delivering Data-Driven Value at Scale,Zhamak Dehghani,2022,3.79,380,40,goodreads,https://www.goodreads.com/book/show/58230358-d...
6,Data Pipelines with Apache Airflow,Bas P. Harenslak,2020,4.31,118,16,goodreads,https://www.goodreads.com/book/show/52778122-d...
7,Spark: The Definitive Guide: Big Data Processi...,Bill Chambers,2018,4.14,283,31,goodreads,https://www.goodreads.com/book/show/38467996-s...
8,Kafka: The Definitive Guide: Real-Time Data an...,Neha Narkhede,2017,4.15,727,81,goodreads,https://www.goodreads.com/book/show/28321010-k...
9,Making Sense of Stream Processing,Martin Kleppmann,2016,4.25,190,30,goodreads,https://www.goodreads.com/book/show/29598815-m...


In [18]:
# Save CSV with 5+ fields (Price replaced by RatingsCount)
df_out = df[["Title", "Author", "Year", "Rating", "RatingsCount", "ReviewsCount", "Source", "URL"]].copy()

df_out.to_csv("techreads_books.csv", index=False, encoding="utf-8")
print("Saved: techreads_books.csv")

df_out.head(15)

Saved: techreads_books.csv


,Title,Author,Year,Rating,RatingsCount,ReviewsCount,Source,URL
0,Fundamentals of Data Engineering: Plan and Bui...,Joe Reis,2022,4.18,927,97,goodreads,https://www.goodreads.com/book/show/61218623-f...
1,Designing Data-Intensive Applications,Martin Kleppmann,2015,4.70,10687,947,goodreads,https://www.goodreads.com/book/show/23463279-d...
2,The Data Warehouse Toolkit: The Complete Guide...,Ralph Kimball,1996,4.18,1035,65,goodreads,https://www.goodreads.com/book/show/748203.The...
3,Data Pipelines Pocket Reference: Moving and Pr...,James Densmore,2021,3.77,242,31,goodreads,https://www.goodreads.com/book/show/55841851-d...
4,Database Internals: A deep-dive into how distr...,Alex Petrov,2019,4.26,558,69,goodreads,https://www.goodreads.com/book/show/44647144-d...
5,Data Mesh: Delivering Data-Driven Value at Scale,Zhamak Dehghani,2022,3.79,380,40,goodreads,https://www.goodreads.com/book/show/58230358-d...
6,Data Pipelines with Apache Airflow,Bas P. Harenslak,2020,4.31,118,16,goodreads,https://www.goodreads.com/book/show/52778122-d...
7,Spark: The Definitive Guide: Big Data Processi...,Bill Chambers,2018,4.14,283,31,goodreads,https://www.goodreads.com/book/show/38467996-s...
8,Kafka: The Definitive Guide: Real-Time Data an...,Neha Narkhede,2017,4.15,727,81,goodreads,https://www.goodreads.com/book/show/28321010-k...
9,Making Sense of Stream Processing,Martin Kleppmann,2016,4.25,190,30,goodreads,https://www.goodreads.com/book/show/29598815-m...
